# Resume the interrupted free adapter comparison
**Status:** both adapters completed this comparison; saved summaries were verified on 2026-10-19. No resume is needed. This notebook is retained for recovery reference.\nUse this only after the original Colab runtime is confirmed stopped (Manage sessions → no active session). Use the same account and an eligible free GPU. The notebook restores the exact code revision from the saved Drive metadata, keeps completed rows, and resumes remaining work. It does not purchase compute or bypass quota limits. If no free GPU is offered, stop and retry another day.
This notebook is specific to `bounded-adapter-comparison-2026-10-17`. It requires the saved Drive folder and both pilot adapters. Do not replace or delete existing predictions to restart scoring.


In [ ]:
from pathlib import Path
import json, re, subprocess, sys, torch
if not torch.cuda.is_available():
    raise RuntimeError("No free CUDA GPU offered. Stop and retry later; do not upgrade.")
from google.colab import drive
drive.mount("/content/drive")
run = Path("/content/drive/MyDrive/Beadform-free-qwen/runs/bounded-adapter-comparison-2026-10-17")
metadata = json.loads((run/"comparison-metadata.json").read_text())
revision = metadata["repo_commit"]
if not re.fullmatch(r"[0-9a-f]{40}",revision):
    raise ValueError("Invalid saved repository revision.")
if torch.cuda.get_device_name(0) != metadata["gpu"]:
    raise RuntimeError("GPU differs from the saved comparison. Use a separate run for a changed configuration.")
repo=Path("/content/beadform")
if not repo.exists():
    subprocess.run(["git","clone","--no-checkout","https://github.com/c-tustin/beadform.git",str(repo)],check=True)
if subprocess.check_output(["git","status","--porcelain"],cwd=repo,text=True).strip():
    # A no-checkout clone is allowed to start without a worktree.
    if (repo/"training").exists():
        raise RuntimeError("The Colab checkout has local changes. Preserve them and use a clean checkout.")
subprocess.run(["git","fetch","--depth","1","origin",revision],cwd=repo,check=True)
subprocess.run(["git","checkout","--detach",revision],cwd=repo,check=True)
assert subprocess.check_output(["git","rev-parse","HEAD"],cwd=repo,text=True).strip()==revision
subprocess.run([sys.executable,"-m","pip","install","-q","-r",str(repo/"training/requirements.txt")],check=True)
print("Restored exact comparison revision:",revision)

# Preserve the original environment record; refuse to mix dependency versions.
from importlib.metadata import version
expected={}
for line in (run/"pip-freeze.txt").read_text().splitlines():
    if "==" in line and not line.startswith("#"):
        name,value=line.split("==",1)
        expected[name.lower().replace("_","-")]=value
critical=("torch","transformers","tokenizers","peft","accelerate","bitsandbytes","lm-format-enforcer","jsonschema")
different={}
for name in critical:
    if name not in expected:
        raise RuntimeError(f"Original version missing for {name}; inspect the saved environment.")
    current=version(name)
    if current != expected[name]:
        different[name]={"saved":expected[name],"current":current}
if different:
    raise RuntimeError("Dependency versions differ. Restore these exact saved versions before resuming: "+json.dumps(different))
original=run/"pip-freeze-before-resume.txt"
if not original.exists():
    original.write_bytes((run/"pip-freeze.txt").read_bytes())
print("Critical dependency versions match the saved run.")


In [ ]:
# Reuse the original comparison code, output paths and resume manifests.
# Completed rows are retained; unfinished rows continue only after matching configuration.
comparison=json.loads((repo/"training/colab_compare_adapters.ipynb").read_text())
exec("".join(comparison["cells"][-1]["source"]))
